## **Arabic Named Entity Recognition**
Named entity recognition (NER) is the task of finding spans of text that constitute proper names and labeling the type of such entities.
As we discussed in the lecture, this task can be formulated as a sequence labeling problem, e.g., using the BIO tagging scheme.
Your task is to build a NER tagger for Arabic using Conditional Random Fields (CRFs) and explore different sets of features to achieve improvement over the baseline tagger.
Arabic is a morphologically rich language, which poses certain challenges to NLP.
We use ANERcorp that has 150K words annotated for four entities: Location (LOC), Organization (ORG), and Person (PER), and Miscellaneous (MISC).
Since ANERcorp only has train and test splits, we take 10% of the train as a development set.


## **The Task**
Your task is to improve the feature function (`word2features()`).
Currently, the baseline model only takes into account previous and next words.
You may use external tools to obtain useful features such as POS and morphological tags.
You will also need to submit a report describing your method, result, and analysis (1-2 pages).

## **Data Preparation**

In [153]:
%cd ~/haozezhao/code/NLP801/assignment1
%pip install scikit-learn seqeval
%pip install python-crfsuite
!test -f ANERcorp-CamelLabSplits.zip || wget https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
!unzip -o ANERcorp-CamelLabSplits.zip -x "__MACOSX/*"

/vast/users/xiaodan/haozezhao/code/NLP801/assignment1
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
--2026-10-18 08:13:26--  https://camel.abudhabi.nyu.edu/anercorp/ANERcorp-CamelLabSplits.zip
Connecting to 172.22.13.145:3128... connected.
Proxy tunneling failed: ForbiddenUnable to establish SSL connection.
unzip:  cannot find or open ANERcorp-CamelLabSplits.zip, ANERcorp-CamelLabSplits.zip.zip or ANERcorp-CamelLabSplits.zip.ZIP.


In [154]:
from sklearn.model_selection import train_test_split

# read ANER named entity dataset
def read_data(file_path):
    with open(file_path, mode='r') as f:
        sent, sents = [], []
        for line in f.readlines():
            if len(line.strip()) > 0:
                # split the line by space
                token = line.strip().split(' ')
                # unpack the token
                word, ner = token
                # append the tuple (word, ner tag) to a list for one sentence
                sent.append((word, ner.replace('PERS', 'PER')))
            # if the line is empty and the list is not empty
            elif len(sent):
                sents.append(sent)
                sent = []
        if sent:
            sents.append(sent)
    return sents

# read training and test data
_train_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_train.txt')
test_sents = read_data('./ANERcorp-CamelLabSplits/ANERCorp_CamelLab_test.txt')

# since this dataset does not have dev set, we take 10% of the train as dev set
train_sents, dev_sents = train_test_split(_train_sents, test_size=0.1, random_state=12345)

## **Feature Engineering**
Here, your task is to design a function that tranforms each word into a set of features.
You may want to use existing tools to get POS, morphological features, etc.
The current model gives `0.70` in $F_1$ on the dev set.

In [155]:
%pip install camel-tools
!camel_data -i light

Note: you may need to restart the kernel to use updated packages.
No new packages will be installed.


In [156]:
from camel_tools.disambig.mle import MLEDisambiguator
disambiguator = MLEDisambiguator.pretrained()

In [157]:
def word2features(sent, i, morph=None):
    word = sent[i][0]

    features = {
        'word': word,
        # 'word.lower': word.lower(),
        # 'word.len': str(len(word)),
    }

    for n in range(1, 6):
        if len(word) >= n:
            features[f'prefix_{n}'] = word[:n]
            features[f'suffix_{n}'] = word[-n:]

    if morph:
        features.update(morph)

        
    for offset in (-1, 1):
        j = i + offset
        if 0 <= j < len(sent):
            features[f'{offset}:word'] = sent[j][0]
        elif offset < 0:
            features[f'BOS{offset}'] = True
        else:
            features[f'EOS{offset}'] = True


    return features

Extract the features from the data and train your model.

In [158]:
import pycrfsuite

def sent2features(sent):
    # return [word2features(sent, i) for i in range(len(sent))]
    words = [token for token, _ in sent]
    analyses = disambiguator.disambiguate(words)

    features = []
    for i in range(len(sent)):
        analysis = analyses[i].analyses[0] if analyses[i].analyses else None

        morph = {}
        if analysis is not None:
            morph['pos'] = analysis.analysis.get('pos', 'UNK')
            for key in ('gen', 'num', 'per', 'stt', 'asp', 'mod'):
                value = analysis.analysis.get(key)
                if value:
                    morph[f'morph_{key}'] = value

        features.append(word2features(sent, i, morph))

    return features

def sent2labels(sent):
    return [label for token, label in sent]

def sent2tokens(sent):
    return [token for token, label in sent]


X_train = [sent2features(s) for s in train_sents]
y_train = [sent2labels(s) for s in train_sents]

X_dev = [sent2features(s) for s in dev_sents]
y_dev = [sent2labels(s) for s in dev_sents]

# create pycrfsuite.Trainer instance
trainer = pycrfsuite.Trainer(verbose=False)

# load the training data
for xseq, yseq in zip(X_train, y_train):
    trainer.append(xseq, yseq)

# training parameteres
trainer.set_params({
    'c1': 1.0,   # coefficient for L1 penalty
    'c2': 1e-3,  # coefficient for L2 penalty
    'max_iterations': 50,  # stop earlier

    # include transitions that are possible, but not observed
    'feature.possible_transitions': True
})

trainer.train('arabic-ner.crfsuite')

## **Evaluation**

Evaluate your model on the development set and the test set.

**Note**: You may run evaluation on the test set **ONLY ONCE** at the end. You should not use the test set to check how well your model is currently performing and modify the model accordingly.
Use the dev set, or create a tune set for this purpose.

In [159]:
# instantiate the trained tagger
tagger = pycrfsuite.Tagger()
tagger.open('arabic-ner.crfsuite')

# tag sentences in the dev set
y_pred_dev = [tagger.tag(xseq) for xseq in X_dev]

In [160]:
from seqeval.metrics import classification_report, f1_score

print(classification_report(y_dev, y_pred_dev, digits=4))
print("Dev F1:", f1_score(y_dev, y_pred_dev))

              precision    recall  f1-score   support

         LOC     0.9121    0.8892    0.9005       397
        MISC     0.7164    0.5517    0.6234        87
         ORG     0.8271    0.6286    0.7143       175
         PER     0.8066    0.8133    0.8099       241

   micro avg     0.8518    0.7856    0.8173       900
   macro avg     0.8156    0.7207    0.7620       900
weighted avg     0.8484    0.7856    0.8133       900

Dev F1: 0.8173410404624278


In [162]:
X_test = [sent2features(s) for s in test_sents]
y_test = [sent2labels(s) for s in test_sents]

y_pred_test = [tagger.tag(xseq) for xseq in X_test]

print(classification_report(y_test, y_pred_test, digits=4))
print("Test F1:", f1_score(y_test, y_pred_test))

output_path = "ANERCorp_CamelLab_test.pred.txt"

with open(output_path, "w", encoding="utf-8") as f:
    for sent, pred_labels in zip(test_sents, y_pred_test):
        for (word, _), label in zip(sent, pred_labels):
            f.write(f"{word} {label}\n")
        f.write("\n")

              precision    recall  f1-score   support

         LOC     0.7838    0.8151    0.7991       676
        MISC     0.7541    0.3786    0.5041       243
         ORG     0.6908    0.3747    0.4859       459
         PER     0.7130    0.5204    0.6017       907

   micro avg     0.7414    0.5632    0.6401      2285
   macro avg     0.7354    0.5222    0.5977      2285
weighted avg     0.7338    0.5632    0.6264      2285

Test F1: 0.6401392688385974
